In [2]:
import os
from dotenv import load_dotenv
import pandas as pd
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Load environment variables
load_dotenv()

# Same collection and dataset behavior as before
COLLECTION_NAME = os.getenv('COLLECTION_NAME')

df = pd.read_excel('../data/merged_with_embeddings.xls')

def extract_channel_thumbnail(val):
    """
    Ensures we always store a single string URL.
    Handles dicts from YouTube API format or direct strings.
    """
    if isinstance(val, dict):
        for quality in ['high', 'medium', 'default']:
            if quality in val and 'url' in val[quality]:
                return val[quality]['url']
        return None
    elif isinstance(val, str):
        return val.strip() if val.strip() else None
    else:
        return None

# Keep the same sanitization logic
df['channel_thumbnail'] = df['channel_thumbnail'].apply(extract_channel_thumbnail)

# Initialize ChromaDB exactly as before
client = chromadb.PersistentClient(path="./chromadb_data1")
collection = client.get_or_create_collection(name="video_embeddings")

ids = df['id'].astype(str).tolist()

def parse_embedding(x):
    if isinstance(x, str):
        return eval(x)
    return x

embeddings = df['embedding'].apply(parse_embedding).tolist()

metadatas = df[['id', 'title', 'description', 'categoryId', 'duration',
                'channel_id', 'channel_title', 'channel_thumbnail',
                'channel_description', 'channel_subscriberCount',
                'likeCount', 'transcript']].to_dict(orient='records')

# Recreate or overwrite same collection
client.delete_collection("video_embeddings")
collection = client.get_or_create_collection(name="video_embeddings")

collection.add(
    ids=ids,
    embeddings=embeddings,
    metadatas=metadatas
)

# LangChain connection wrapper for the same underlying Chroma collection
# This keeps the storage and metadata flow unchanged while enabling LangChain usage
langchain_store = Chroma(
    collection_name='video_embeddings',
    embedding_function=None,
    client=client,
)

# Build documents using the same metadata structure for compatibility
documents = [
    Document(
        page_content=row['title'] + ' ' + str(row['transcript']),
        metadata={
            'id': row['id'],
            'title': row['title'],
            'description': row['description'],
            'categoryId': row['categoryId'],
            'duration': row['duration'],
            'channel_id': row['channel_id'],
            'channel_title': row['channel_title'],
            'channel_thumbnail': row['channel_thumbnail'],
            'channel_description': row['channel_description'],
            'channel_subscriberCount': row['channel_subscriberCount'],
            'likeCount': row['likeCount'],
            'transcript': row['transcript'],
        }
    )
    for _, row in df.iterrows()
]

print("✅ Embeddings + cleaned channel thumbnails stored in ChromaDB!")
print("LangChain Chroma wrapper initialized successfully.")

✅ Embeddings + cleaned channel thumbnails stored in ChromaDB!
LangChain Chroma wrapper initialized successfully.


C:\Users\Dhruvin\AppData\Local\Temp\ipykernel_35884\1146604076.py:64: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  langchain_store = Chroma(
